# AI-Based Business Location Selector
## Notebook 02: Supervised Model Training, Clustering & MCDA Suitability Scoring

**Domain:** Machine Learning & Geospatial Analytics  
**Objective:** Compare predictive regression models (Linear Regression Baseline vs. Decision Tree vs. Random Forest), perform K-Means spatial clustering, and apply multi-criteria ranking.


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.cluster import KMeans


### 1. Load Data & Feature Engineering
We engineer interaction features: `rent_to_footfall`, `competitor_density`, and facility indicators.


In [ ]:
data_path = os.path.join('..', 'data', 'business_locations.csv')
df = pd.read_csv(data_path)

# Feature engineering
df['rent_to_footfall'] = df['monthly_rent'] / (df['daily_footfall'] + 1.0)
df['competitor_density'] = df['competitor_count'] / (df['population'] / 10000.0)
df['access_composite'] = (df['accessibility_score'] * 0.4) + (df['parking_score'] * 0.3) + (df['public_transport_score'] * 0.3)
df['has_offices'] = df['nearby_facilities'].str.contains('Offices', case=False).astype(int)
df['has_colleges'] = df['nearby_facilities'].str.contains('Colleges', case=False).astype(int)
df['has_hospitals'] = df['nearby_facilities'].str.contains('Hospitals', case=False).astype(int)
df['has_transit'] = df['nearby_facilities'].str.contains('Transit Hub|Metro', case=False).astype(int)
print('Feature engineering complete. Sample features:')
df[['rent_to_footfall', 'competitor_density', 'access_composite']].head()


### 2. Preprocessing & Train-Test Split (80/20)
We construct a scikit-learn `ColumnTransformer` to handle scaling and encoding cleanly without data leakage.


In [ ]:
target = 'target_demand'
X = df.drop(columns=[target, 'monthly_sales', 'business_potential', 'location_id', 'location_name', 'neighborhood', 'city', 'nearby_facilities'])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

num_features = [
    'monthly_rent', 'population', 'competitor_count', 'daily_footfall',
    'accessibility_score', 'parking_score', 'public_transport_score',
    'rent_to_footfall', 'competitor_density', 'access_composite',
    'has_offices', 'has_colleges', 'has_hospitals', 'has_transit'
]
ord_features = ['income_level']
nom_features = ['business_type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('ord', OrdinalEncoder(categories=[['Low', 'Medium', 'High']]), ord_features),
        ('nom', OneHotEncoder(drop='first', sparse_output=False), nom_features)
    ]
)
print(f'Training shape: {X_train.shape}, Test shape: {X_test.shape}')


### 3. Model Training & Comparison
As mandated by Section 7 & 14 of the guide, we compare multiple algorithms:
- **Linear Regression** (Interpretable baseline)
- **Decision Tree Regressor** (Rule-based non-linear baseline)
- **Random Forest Regressor** (Ensemble non-linear regressor)


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=6, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=9, random_state=42, n_jobs=-1)
}

results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', model)])
    
    # 5-Fold Cross Validation
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2')
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'R2 Score': round(r2, 4),
        '5-Fold CV R2': f'{cv_scores.mean():.4f} +/- {cv_scores.std():.4f}'
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results)
results_df


### 4. Residual and Actual vs. Predicted Plots
Visually inspect prediction error distributions for the champion Random Forest model.


In [ ]:
rf_pipe = fitted_models['Random Forest']
y_test_pred = rf_pipe.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Actual vs Predicted
axes[0].scatter(y_test, y_test_pred, alpha=0.6, color='#2980b9')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', lw=2)
axes[0].set_title('Actual vs. Predicted Demand (Test Set)')
axes[0].set_xlabel('Actual Demand')
axes[0].set_ylabel('Predicted Demand')

# Residual distribution
residuals = y_test - y_test_pred
sns.histplot(residuals, kde=True, ax=axes[1], color='#e74c3c')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Residual Error Distribution (y_test - y_pred)')
axes[1].set_xlabel('Residual Error')
plt.tight_layout()
plt.show()


### 5. Feature Importance Analysis
Explainable AI: Determine the top drivers of customer demand.


In [ ]:
rf_reg = rf_pipe.named_steps['reg']
ohe = rf_pipe.named_steps['prep'].named_transformers_['nom']
nom_cols_encoded = list(ohe.get_feature_names_out(nom_features))
all_features = num_features + ord_features + nom_cols_encoded
importances = rf_reg.feature_importances_

fi_df = pd.DataFrame({'feature': all_features, 'importance': importances}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df.head(10), x='importance', y='feature', palette='viridis')
plt.title('Top 10 Feature Importances (Random Forest Regressor)')
plt.xlabel('Gini Importance')
plt.show()


### 6. Unsupervised Learning: K-Means Clustering
Group candidate locations based on multi-dimensional attributes. Determine optimal $k$ using the Elbow Method and Silhouette Analysis.


In [ ]:
cluster_cols = ['monthly_rent', 'daily_footfall', 'population', 'target_demand', 'accessibility_score', 'competitor_count']
scaler = StandardScaler()
X_clust_scaled = scaler.fit_transform(df[cluster_cols])

k_range = list(range(2, 8))
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_clust_scaled, km.labels_))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(k_range, inertias, 'bo-', label='Inertia (Elbow Method)')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (SSE)', color='b')
ax2 = ax1.twinx()
ax2.plot(k_range, silhouettes, 'rs--', label='Silhouette Score')
ax2.set_ylabel('Silhouette Score', color='r')
plt.title('Cluster Evaluation: Elbow Method and Silhouette Score')
plt.show()


In [ ]:
# Fit optimal K=4
optimal_k = 4
final_km = KMeans(n_clusters=optimal_k, random_state=42, n_init=15)
df['cluster_id'] = final_km.fit_predict(X_clust_scaled)

# Cluster Centroid Profiles
cluster_profiles = df.groupby('cluster_id')[cluster_cols].mean().round(1)
print('Cluster Centroid Profiles:')
cluster_profiles


### 7. Multi-Criteria Decision Analysis (MCDA) Suitability Scoring
We implement the transparent formula from Section 9 of the guide:
$$S = 0.30D + 0.20A + 0.15F + 0.15P + 0.10C + 0.10R$$
All sub-scores are normalized to [0, 100].


In [ ]:
# Normalize sub-components
D = (df['target_demand'] / df['target_demand'].max()) * 100.0
A = (df['accessibility_score'] / 10.0) * 100.0
F = (df['daily_footfall'] / df['daily_footfall'].max()) * 100.0
P = (((df['parking_score'] * 0.5) + (df['public_transport_score'] * 0.5)) / 10.0) * 100.0
C = (1.0 - (df['competitor_count'] / 15.0)).clip(0, 1) * 100.0
budget = 50000
R = df['monthly_rent'].apply(lambda r: 100.0 * max(0.0, 1.0 - (r / (budget * 1.3))) if r <= budget else max(0.0, 30.0 * (1.0 - (r - budget) / budget)))

# Weighted Composite Formula
df['suitability_score'] = (0.30*D + 0.20*A + 0.15*F + 0.15*P + 0.10*C + 0.10*R).round(1)
df['rank'] = df['suitability_score'].rank(ascending=False).astype(int)

# Display Top 5 Ranked Candidates for Cafe
cafe_top5 = df[df['business_type'] == 'Cafe'].sort_values('suitability_score', ascending=False).head(5)
print('Top 5 Ranked Candidate Locations for Cafe:')
cafe_top5[['rank', 'location_name', 'neighborhood', 'monthly_rent', 'daily_footfall', 'target_demand', 'suitability_score']]
